# Py-optram tutorial

This notebook introduces Py-optram for someone using the package for the first time. It follows the main workflow from an area of interest to a fitted OPTRAM trapezoid, then explains how the saved results can support later analysis.

Most cells work locally. The cell that runs `optram()` is clearly marked because it connects to the Copernicus Data Space Ecosystem (CDSE) and downloads Sentinel-2 data. The example area and dates are placeholders: replace them with values suitable for your study.

## 1. What Py-optram is

Py-optram is a Python implementation of the core OPTRAM workflow. It can acquire Sentinel-2 inputs from CDSE, combine vegetation-index and SWIR Transformed Reflectance values in a pixel table, and fit the wet and dry edges of the resulting point cloud.

The package is developed with behavioral parity with the R package rOPTRAM as a primary goal. Py-optram does not yet reproduce every rOPTRAM feature, so this tutorial concentrates on the supported CDSE workflow.

## 2. The workflow in plain language

For every usable image pixel, OPTRAM relates a vegetation index (VI) to SWIR Transformed Reflectance (STR). Observations collected across multiple dates form a cloud of points. Py-optram estimates the wet and dry boundaries of that cloud and reports the fitting error.

The main sequence is:

1. Define an area of interest and a date range.
2. Download VI and STR rasters from CDSE.
3. Build `VI_STR_data.parquet`, with one row per retained pixel observation.
4. Fit the wet and dry trapezoid edges.
5. Return the edge-fitting RMSE and save the fitting results.

Soil-moisture calculation is a separate optional stage. The high-level `optram()` wrapper stops after edge fitting and does **not** create soil-moisture rasters.

## 3. Installation

Py-optram requires **Python 3.11 or newer**. Using a virtual environment is recommended so that the package and its geospatial dependencies are isolated from other projects.

For normal use, install the current GitHub version from a terminal:

```bash
python -m pip install git+https://github.com/nivbenar/Py-optram.git
```

Developers who want to edit the source can instead clone the repository and install it in editable mode:

```bash
git clone https://github.com/nivbenar/Py-optram.git
cd Py-optram
python -m pip install -e .
```

Restart the notebook kernel after installation if the import below is not immediately available.

## 4. Imports

The tutorial uses `Path` for readable output paths, GeoPandas for the area of interest, pandas for the saved Parquet table, and Shapely to create a simple example polygon. Py-optram is imported as `op` to keep later calls short.

This cell does not use the network.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import box

import pyoptram as op

## 5. Create a GeoDataFrame area of interest

The public Py-optram acquisition functions require the area of interest, or AOI, to be a non-empty `geopandas.GeoDataFrame` containing Polygon or MultiPolygon geometry. The GeoDataFrame must also have a coordinate reference system (CRS).

A bounding-box list, a filename string, or a raw GeoJSON dictionary is not a valid AOI for this API. The coordinates below create a demonstration polygon near Yuma, Arizona. They are only placeholders.

In [ ]:
aoi = gpd.GeoDataFrame(
    geometry=[box(-114.75, 32.55, -114.45, 32.75)],
    crs="EPSG:4326",
)

Before downloading anything, display the AOI and confirm that it has a CRS. In a Jupyter environment, evaluating the GeoDataFrame also provides a quick visual and tabular check of the geometry. This is an offline step.

In [ ]:
print(aoi.crs)
aoi

### Load an AOI from a GIS file

For a real project, you will usually load an existing GeoPackage, Shapefile, or GeoJSON file instead of typing coordinates. GeoPandas returns the required GeoDataFrame:

```python
aoi = gpd.read_file("path/to/aoi.gpkg")
```

Run that line in place of the example-polygon cell. Check `aoi.crs` after loading the file; a missing CRS must be corrected before acquisition.

## 6. CDSE credentials

CDSE credentials are needed for the download step. Never place real credentials in a notebook that will be committed or shared. Store them once using placeholders replaced only in your private session:

```python
op.store_cdse_credentials(
    client_id="YOUR_CDSE_CLIENT_ID",
    client_secret="YOUR_CDSE_CLIENT_SECRET",
)
```

Alternatively, set the `OAUTH_CLIENTID` and `OAUTH_SECRET` environment variables and call `op.store_cdse_credentials()` without arguments. Stored credentials can be reused in later Python or Jupyter sessions. Access tokens are requested when needed and are not persisted by Py-optram.

## 7. Basic package options

`optram_options()` controls session-level settings used by the workflow. This tutorial uses NDVI as its example vegetation index, but Py-optram can also acquire SAVI or MSAVI. Here we select NDVI and request only the VI and STR products, which skips the additional bottom-of-atmosphere raster download.

Other useful options include `max_cloud`, `SWIR_band`, `scm_mask`, and `trapezoid_method`. Start with the defaults unless your analysis requires a specific change. Option calls are silent by default; `show_opts=True` prints the current settings. This cell is offline.

In [ ]:
op.optram_options("veg_index", "NDVI")
op.optram_options("only_vi_str", True)
op.optram_options(show_opts=True)

## 8. Define output folders

Use separate folders for downloaded Sentinel-2 rasters and tabular fitting results. Explicit paths make it easier to find the outputs and continue working in a later session. If these folders do not exist, the workflow creates them as needed.

This cell only defines paths; it does not create or download anything.

In [ ]:
s2_output_dir = Path("tutorial_output/sentinel2")
data_output_dir = Path("tutorial_output/results")

## 9. Run the high-level `optram()` wrapper

The wrapper is the easiest way to run the supported workflow. It accepts the AOI GeoDataFrame, a date range written as `YYYY-MM-DD`, and the two output directories. The end date must be later than the start date.

The example dates are placeholders. Choose a period appropriate for your study and expect the download time to vary with AOI size, date range, cloud filtering, and the number of available scenes.

> **Network-dependent cell:** The next cell authenticates with CDSE, searches the catalog, and downloads raster data. It requires valid stored credentials and internet access. It can also take significant time. Skip it if you are reading the tutorial without credentials or already have outputs from an earlier run.

`optram()` prints and returns a one-row DataFrame containing wet- and dry-edge RMSE values. It stops at edge fitting and does **not** calculate soil moisture.

In [ ]:
rmse = op.optram(
    aoi=aoi,
    from_date="2024-01-01",
    to_date="2024-03-31",
    s2_output_dir=s2_output_dir,
    data_output_dir=data_output_dir,
)

rmse

## 10. What `optram()` does internally

The wrapper connects several public functions in a fixed sequence. It calls `acquire_optram_inputs()` to download imagery, finds the configured VI rasters and STR rasters, and passes them to `optram_ndvi_str()`. That function builds the pixel table and saves it as Parquet. Finally, `optram_wetdry_coefficients()` fits the wet and dry edges.

The returned RMSE values describe the edge fits. They are not soil-moisture estimates. Lower-level functions are available when you need direct control over one stage, but beginners can start with the wrapper.

## 11. Output directory structure

With `only_vi_str=True` and `veg_index="NDVI"`, the main outputs have this structure:

```text
tutorial_output/
├── sentinel2/
│   ├── NDVI/
│   │   └── NDVI_*.tif
│   └── STR/
│       └── STR_*.tif
└── results/
    ├── VI_STR_data.parquet
    ├── trapezoid_points.csv
    ├── wetdry_coefficients.csv
    └── wetdry_rmse.csv
```

If `only_vi_str=False`, acquisition also creates a `BOA/` directory. The Parquet file contains paired pixel observations; the CSV files contain fitted edge points, coefficients, and fitting errors.

## 12. Read `VI_STR_data.parquet`

Parquet preserves data types and is efficient for a potentially large pixel table. pandas can read the file directly. This step does not use the network, but the file must already exist from a successful wrapper run.

In [ ]:
vi_str = pd.read_parquet(
    data_output_dir / "VI_STR_data.parquet"
)

vi_str.head()

## 13. Understand `VI` and `STR`

This tutorial uses NDVI, but Py-optram can also use SAVI or MSAVI. The saved VI-STR table always names the scientific vegetation-index column **`VI`**, not `NDVI`, regardless of which of these indices was selected. Use `vi_str["VI"]` in later analysis.

`STR` is SWIR Transformed Reflectance. Other columns record coordinates, time, tile information, source files, and pixel provenance. Looking at the two central columns is a simple first check that the table contains usable values.

In [ ]:
vi_str[["VI", "STR"]].head()

## 14. Fit the wet and dry edges from the table

The wrapper already fitted the edges and saved its results. Running the fitting function directly is useful when you want the RMSE, coefficient table, and fitted edge points as Python objects.

`return_outputs=True` returns those three objects in that order. The default input column names are `VI` and `STR`, so no column arguments are needed here. This is an offline calculation using the table already loaded into memory.

In [ ]:
rmse, coefficients, edges = op.optram_wetdry_coefficients(
    vi_str,
    return_outputs=True,
)

rmse

## 15. Plot the VI-STR cloud

The plot places vegetation-index values on the horizontal axis and STR values on the vertical axis. The fitted dry edge is shown in orange and the fitted wet edge in blue. Edge points are included by default.

This cell is offline. It uses the table and fitted edges created above. For a dense table, you can later try `plot_colors="density"`, but the default plot keeps the first example simple.

In [ ]:
op.plot_vi_str_cloud(
    vi_str,
    edges,
)

## 16. Soil moisture is a separate optional stage

The high-level `optram()` wrapper deliberately stops after fitting and RMSE reporting. To create soil-moisture rasters, use `optram_calculate_soil_moisture()` separately with correctly paired VI and STR raster paths plus fitted coefficients.

Each VI/STR pair must describe the same raster grid: shape, transform, and CRS must match. The function writes float32 `SM_*.tif` files. Soil porosity defaults to 0.4, and results are not clipped unless `clip=True` is requested.

File pairing deserves careful attention and is project-specific, so this beginner notebook does not add complicated matching logic. See `help(op.optram_calculate_soil_moisture)` and the repository README when you are ready for this optional stage.

## 17. High-level and lower-level functions

Use `optram()` when you want the complete supported acquisition-to-fitting workflow. The lower-level API lets you repeat or customize one part without rerunning everything:

| Goal | Public function |
|---|---|
| Acquire VI and STR rasters | `acquire_optram_inputs()` |
| Build the pixel table | `optram_ndvi_str()` |
| Fit wet and dry edges | `optram_wetdry_coefficients()` |
| Plot the VI-STR cloud | `plot_vi_str_cloud()` |
| Calculate a soil-moisture array | `calculate_soil_moisture()` |
| Write soil-moisture rasters | `optram_calculate_soil_moisture()` |

A good beginner workflow is to run the wrapper once, then use the saved Parquet and CSV outputs for offline exploration.

## 18. Troubleshooting: AOI and credentials

**`aoi must be a geopandas.GeoDataFrame`** means that a list, filename, or other object was passed instead of a GeoDataFrame. Load the GIS file with `gpd.read_file()` or create a GeoDataFrame as shown above. Also confirm that the GeoDataFrame is not empty and that `aoi.crs` is defined.

**Credential or authentication errors** usually mean that stored credentials are missing, incomplete, or rejected by CDSE. Store them privately before running the network cell. Do not replace the notebook’s placeholders with real secrets and then commit the notebook.

## 19. Troubleshooting: data and fitting

**No scenes or missing output rasters:** check the AOI, date range, cloud threshold, and credentials. A short period or strict filtering can leave no usable scenes.

**Missing `VI_STR_data.parquet`:** confirm that the network cell completed successfully and that you are reading from the same `data_output_dir`.

**A missing `NDVI` column:** use `VI`. The table always uses the generic vegetation-index column name.

**No valid rows or insufficient edge points:** the AOI or date range may not provide enough usable observations, or filtering may be too strict. A larger sample is often needed for stable trapezoid fitting.

## 20. Summary

You have now seen the complete beginner path:

- Py-optram requires Python 3.11 or newer.
- The AOI must be a GeoPandas GeoDataFrame with polygon geometry and a CRS.
- CDSE credentials and network access are required for acquisition.
- `optram()` downloads inputs, creates `VI_STR_data.parquet`, fits the trapezoid edges, and returns RMSE.
- The Parquet table uses `VI`, not `NDVI`; `VI` may contain NDVI, SAVI, or MSAVI values.
- The saved table can be read, fitted, and plotted without downloading the imagery again.
- Soil-moisture calculation is a separate optional stage.

For additional options and lower-level workflows, consult the repository README and the public function documentation with `help(op.<function_name>)`.